# Kontrola běhu — Blood Bowl A/B na Colabu

**Tenhle notebook noc NESPOUŠTÍ.** Odpovídá na otázku, jestli se pustit *smí* — a odpovídá
na ni **před** během, ne ráno po něm.

Logika je ve skriptu `colab_night_preflight.py` v repu, ne v těchhle buňkách. Důvod: buňka
v notebooku se dá otestovat jedině v Colabu, skript se dá otestovat kdekoli. Tenhle notebook
byl ověřen na laptopu (8 jader) — čísla níž vyjdou jiná, o to právě jde.

## Co se kontroluje
1. **stroj** — jader, RAM, commit *(2 vCPU ⇒ ~4× delší běh než 8)*
2. **binárky a ABI** — ⭐ vztah `harness` × `libbb_engine.so`; na tomhle páru umřela fáze B 24. 08.
3. **soubory mimo git** — ⛔ `weights_policy.json` a `rules_bb2016.txt` jsou v `.gitignore`, takže
   je **klon nepřinese** a harness bez nich skončí `rc=1`
4. **zadání** — povinná nulová kontrola, existence předregistrace
5. **sonda 1 páru** v cílovém režimu — běží a tiskne `MOVED WITHOUT THE ARM ACTING`?
6. **tempo změřené ZDE** — manifest zakazuje ho převzít odjinud
7. **rozvrh do sezení** — na kolik stejně velkých kusů se to musí nakrájet

In [ ]:
# 1. Prostředí a build  (~4 min)
REF = 'main'          # pro B2: 'b2-wrestle-pricing' -- a napřed do ní mergovat main
%cd /content
!rm -rf /content/bloodbowl
!apt-get update -qq && apt-get install -y -qq cmake g++ > /dev/null 2>&1
!pip install -q pybind11 numpy
!git clone -q https://github.com/jansekera/bloodbowl.git
%cd /content/bloodbowl
!git checkout -q {REF} && git log -1 --format='%h %ad %s' --date=short

pybind_dir = !python3 -c "import pybind11; print(pybind11.get_cmake_dir())"
!cmake -S engine -B engine/build -DCMAKE_BUILD_TYPE=Release -DBUILD_PYTHON=ON \
       -Dpybind11_DIR={pybind_dir[0]} -DCMAKE_CXX_FLAGS="-O3" > /dev/null 2>&1
!cmake --build engine/build -j$(nproc) 2>&1 | tail -2

# ⛔ Harness se NESMÍ zapomenout: když je starší než .so, spadne to na SEGFAULT
#    a hláška vypadá jako chybějící řádek. Přesně tak umřela fáze B 24.08.
!g++ -O2 -std=c++20 -Iengine/include -Iengine/third_party \
     diag_f1_cage_advance_harness.cpp -Lengine/build -lbb_engine \
     -Wl,-rpath,$PWD/engine/build -o diag_f1_cage_advance
!nproc && free -g | head -2

## 2. Soubory, které git nenese

`.gitignore` drží mimo repo dva soubory, **bez kterých se noc nespustí**:

| soubor | proč chybí | co se stane bez něj |
|---|---|---|
| `weights_policy.json` | `/weights*.json` | harness skončí `rc=1` |
| `rules_bb2016.txt` | ř. 68 | nejde ověřit citace pravidel |

Na serveru to nikdo nepoznal — ležely tam od začátku. **Colab klonuje vždycky.**
Nahraj je z Disku (nebo `files.upload()`).

In [ ]:
# 2. Dotáhnout, co git nenese
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/bloodbowl/weights_policy.json /content/bloodbowl/ 2>/dev/null || echo "⛔ weights_policy.json nenalezen na Disku"
!cp /content/drive/MyDrive/bloodbowl/rules_bb2016.txt   /content/bloodbowl/ 2>/dev/null || echo "⚠️ rules_bb2016.txt nenalezen (blokuje jen ověřování citací)"
!ls -l weights_policy.json rules_bb2016.txt 2>&1 | tail -2

## 3. Vlastní kontrola

`SESSION_HOURS` nastav na **skutečný** limit sezení. Zkušenost projektu ze `SERVER_WORKFLOW.md`
mluví o **8–10 h**; ověř si to pro svůj účet, tohle číslo rozhoduje o krájení.

⛔ **B2 z manifestu (`2:dw-dw:1`) nemá nulový matchup** — od 27. 08. má Wrestle každý tým,
takže matchup, kde se rameno spustit *nemůže*, neexistuje. `run_night_ab.sh` takový běh
odmítne (`exit 2`). Řešením je `CONTROL_MODE2=1`, tedy krátká noha v mode 2, kde mají obě
ramena touž konfiguraci a delta musí vyjít přesně 0.

In [ ]:
# 3. Kontrola běhu
MODE           = 11
MATCHUPS       = '2:dw-dw:1'
PREREG         = 'evidence/night_prereg_20260829_b2.preds'
PAIRS          = 4800
SESSION_HOURS  = 8.0
WORKERS        = 2          # ⚠️ ne víc než jader; 26.08. 16 workerů na 12 jader nedoběhlo NIC

!python3 colab_night_preflight.py \
    --mode {MODE} --matchups "{MATCHUPS}" --prereg {PREREG} \
    --pairs {PAIRS} --workers {WORKERS} --session-hours {SESSION_HOURS} \
    --tempo-pairs 2 --control-mode2

## 4. Co s výsledkem

* **⛔ STOP** — nespouštět. Každý STOP má v textu, čím se odstraní.
* **⚠️ WARN** — rozhodnutí, ne závada. Typicky *„na jedno sezení to nestačí"*.
* **✅** — teprve pak dává smysl pustit `run_night_ab.sh` s doporučeným `CHUNKS`.

### Krájení přes víc sezení — dvě podmínky, které se nesmí porušit
1. **Kusy stejně velké.** `night_summarize` váží shardy stejně; nestejné kusy rozbijí sdruženou SE.
2. **Engine se mezi kusy nepřestavuje.** Jedna noc = jedno měření, jen roztažené přes víc sezení —
   týž commit, tatáž binárka. Sáhnout mezitím na `.so` znamená měřit dvě různé hry.

⚠️ **Neověřeno a je to první věc k vyzkoušení:** jestli běh naváže na hotové kusy.
Semafory `AB_DONE` a `OK` v adresáři shardu tomu nasvědčují, ale nikdo to nezkusil —
a než se na to spolehne celá noc, má se to ověřit na dvou krátkých kusech.